# Second FABRIC site pair (NCSA <-> TOKY): moved out of `05_scale_up_tmr_n100_and_keys_5_9`

These cells were moved **verbatim** from `notebooks/core/05_scale_up_tmr_n100_and_keys_5_9.ipynb` (Section B, "Second FABRIC site pair, Feedback #8") on 2026-10-27, because it is not yet established that the paper reports a second site. If it does, move them back and add the `PAPER_TRACEBACK.md` rows.

* Original cell indices in `05_scale_up_tmr_n100_and_keys_5_9` (before the move): 7, 9, 10, 14, 15, 16, 18, 20, 26-35. Cell 8 (shared setup) stayed in the core notebook; a copy is the first code cell below because later cells here need its imports and `RESULTS`/`fablib`.
* Cells 11-13, 17 and 19 of that notebook (keys 5-9 recovery and the 10-key `key_pairs_metadata.csv`) are **not** second-site work and stayed in core.
* Results still on disk: `results/data/sdc_site2_toeplitz_finalkey_doseresponse.csv` and `*_site2.json`.
* Needs a live FABRIC slice to run the collection cells; the analysis cells read the CSVs above.


In [1]:
# (copy of cell 8 of 05_scale_up_tmr_n100_and_keys_5_9: shared setup)
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade.sweep_utils import run_real_channel_trial, collect_key_pairs

RESULTS = PROJECT_DIR / "results"
fablib = deploy.get_fablib()

# --- Sanity-check capacity before provisioning; adjust SITE_BOB2 based on what's free ---
for site in ["TOKY", "UTAH", "FIU", "UCSD", "MICH", "CLEM", "STAR", "SALT"]:
    try:
        print(site, fablib.get_site_attribute(site, "cores_available"))
    except Exception as e:
        print(site, "unavailable:", e)


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


TOKY unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UTAH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
FIU unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UCSD unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
MICH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
CLEM unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
STAR unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
SALT unavailable: 'FablibManager' object has no attribute 'get_site_attribute'


## Section B — Second FABRIC site pair (Feedback #8)

Your original pair was **NCSA <-> EDUKY** (Illinois <-> Kentucky): geographically close,
deliberately picked as "clear, robust, minimal noise." To test whether the mock-vs-real
non-effect (Section 4.3 / CMH p=0.16, p=0.32) generalizes, or was itself a property of that
particular low-noise link, we want a pair with **more distance and more hops**.

FABRIC's site list (per the project's public site info) includes coast-adjacent sites such
as `UTAH`, `MASS` (MGHPCC), `FIU`, `UCSD`, `MICH`, and `CLEM`, alongside the ESnet backbone
production sites (`STAR`/Starlight, `SALT`, `DALL`). Check live capacity before committing —
site availability on FABRIC fluctuates — with `fablib.list_sites()`.

**Suggested pair:** keep Alice + the BMv2 switch at `NCSA` (so Section A's key/switch config
carries over) and move Bob to `UTAH` — a genuinely different coast, more router hops, and a
different jitter profile than the NCSA-EDUKY link. Swap `SITE_BOB2` for whatever
`list_sites()` shows has capacity if `UTAH` doesn't.

In [3]:
SLICE_NAME_2 = 'qfabric-bb84-site2'
SITE_ALICE_2 = 'NCSA'
SITE_BOB_2   = 'TOKY'   # <- change based on the capacity check above if needed
SITE_SW_2    = 'NCSA'
SCENARIO     = 'validation/scenarios/fabric_1km.yml'
BMV2_IMAGE   = 'ghcr.io/kthare10/qfabric-bmv2:latest'

try:
    slice_obj_2 = fablib.get_slice(name=SLICE_NAME_2)
    print(f"Reusing existing slice '{SLICE_NAME_2}' (state: {slice_obj_2.get_state()})")
except Exception:
    slice_obj_2 = deploy.create_slice(fablib, SLICE_NAME_2, SITE_ALICE_2, SITE_BOB_2, SITE_SW_2)

deploy.upload_project(slice_obj_2)
import os
os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE
deploy.install_deps(slice_obj_2, build_bmv2=False)
deploy.setup_switch_docker(slice_obj_2, BMV2_IMAGE)

threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
alice_mac_2, bob_mac_2, sw_alice_mac_2, sw_bob_mac_2, _, _ = deploy.configure_switch(slice_obj_2, threshold)
alice_ip_2, bob_ip_2 = deploy.setup_dataplane_ips(slice_obj_2, alice_mac_2, bob_mac_2)

alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")
print(f"\nSite-2 slice ready: {SITE_ALICE_2} <-> {SITE_BOB_2}, switch at {SITE_SW_2}")


Reusing existing slice 'qfabric-bb84-site2' (state: Configuring)

=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...


ValidationError: Invalid management IP: None

In [5]:
try:
    slice_obj_2 = fablib.get_slice(name=SLICE_NAME_2)
    state = slice_obj_2.get_state()
    print(f"Found existing slice '{SLICE_NAME_2}', state: {state}")
    if state not in ("StableOK", "Active"):  # adjust if your fablib version reports differently
        raise RuntimeError(f"Slice exists but isn't active (state={state}) -- treating as expired")
except Exception as e:
    print(f"Existing slice unusable ({e}), creating fresh...")
    try:
        # Clean up the dead record first so the name is free
        slice_obj_2.delete()
    except Exception:
        pass
    slice_obj_2 = deploy.create_slice(fablib, SLICE_NAME_2, SITE_ALICE_2, SITE_BOB_2, SITE_SW_2)

deploy.upload_project(slice_obj_2)

Found existing slice 'qfabric-bb84-site2', state: StableOK

=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [8]:
import sys
from pathlib import Path
import pandas as pd

# --- Diagnose cwd/PROJECT_DIR first -- this is almost certainly the actual issue ---
print("cwd:", Path.cwd())
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("PROJECT_DIR resolved to:", PROJECT_DIR)
RESULTS = PROJECT_DIR / "results"
print("RESULTS resolved to:", RESULTS, "  exists:", RESULTS.exists())

sys.path.insert(0, str(PROJECT_DIR))
from qne.cascade.key import key_from_sifted_json


def load_key_pairs(metadata_csv):
    """Reload already-collected key pairs from disk -- no FABRIC slice needed.
    Accepts a Path OR a bare filename; if given a bare filename (or a path
    that doesn't exist as given), searches a few likely locations relative
    to PROJECT_DIR before giving up, and reports every path it tried."""
    metadata_csv = Path(metadata_csv)
    candidates = [
        metadata_csv,                                   # exactly what was passed
        RESULTS / "data" / metadata_csv.name,                     # results/<name>, in case a bare
                                                           # filename (no dir) was passed
        Path.cwd() / metadata_csv.name,                   # current working dir
        PROJECT_DIR / metadata_csv.name,                  # project root itself
    ]
    resolved = next((c for c in candidates if c.exists()), None)
    if resolved is None:
        tried = "\n  ".join(str(c.resolve()) for c in candidates)
        raise FileNotFoundError(
            f"Could not find '{metadata_csv.name}'. Tried:\n  {tried}\n"
            f"Run the key-collection cell again if this file was never written, "
            f"or pass the correct path explicitly."
        )
    print(f"loading {resolved}")

    df = pd.read_csv(str(resolved))
    keys = {}
    for _, row in df.iterrows():
        i = int(row["index"])
        # alice_path/bob_path in the CSV are absolute paths from when the CSV was
        # written -- if the results/ folder moved since, rebase to RESULTS instead.
        a_path = Path(row["alice_path"])
        b_path = Path(row["bob_path"])
        if not a_path.exists():
            a_path = RESULTS / "data" / a_path.name
        if not b_path.exists():
            b_path = RESULTS / "data" / b_path.name

        alice_key, alice_idx = key_from_sifted_json(str(a_path), "alice_bits")
        bob_key, bob_idx = key_from_sifted_json(str(b_path), "bob_bits")
        assert alice_idx == bob_idx, f"key {i}: matching_indices mismatch between alice/bob files"
        keys[i] = {"alice_key": alice_key, "bob_key": bob_key,
                   "k_pe": int(row["k_pe"]), "qber": float(row["qber"]), "n_bits": int(row["n_bits"])}
    return df, keys


# --- Site 1 (original + Section C's 5 extra keys, indices 0-9) ---
key_pairs_df, key_pairs = load_key_pairs("key_pairs_metadata.csv")
print(f"site 1: loaded {len(key_pairs)} keys, indices {sorted(key_pairs.keys())}")

# --- Site 2 (NCSA<->UTAH, indices 0-4) ---
key_pairs_df_2, key_pairs_2 = load_key_pairs("key_pairs_metadata_site2.csv")
print(f"site 2: loaded {len(key_pairs_2)} keys, indices {sorted(key_pairs_2.keys())}")a

cwd: /home/fabric/work/qkd-dependability/notebooks
PROJECT_DIR resolved to: /home/fabric/work/qkd-dependability
RESULTS resolved to: /home/fabric/work/qkd-dependability/results   exists: True


FileNotFoundError: Could not find 'key_pairs_metadata.csv'. Tried:
  /home/fabric/work/qkd-dependability/notebooks/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/results/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/notebooks/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/key_pairs_metadata.csv
Run the key-collection cell again if this file was never written, or pass the correct path explicitly.

In [7]:
CSV_SITE2 = RESULTS / "data" / "sdc_site2_toeplitz_finalkey_doseresponse.csv"


In [5]:
# --- Same dose-response sweep as your paper's real-channel data (Section 3.5 /
# results/sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv),
# run on the site-2 link instead. Lower N_RUNS_PER_PROB from the paper's n=150/n=500
# to start -- each point is a live FABRIC round trip; raise it once you've confirmed
# the pipeline runs cleanly end-to-end.
#
# NOTE: we upload each key under its OWN remote filename (alice/bob_sifted_bits_key{i}.json)
# and pass those paths straight to run_real_channel_trial via bob_key_json/alice_key_json,
# instead of re-uploading into the generic "results/alice_sifted_bits.json" name every
# loop iteration. That generic file is also written natively on the node by
# deploy.run_bb84() during collect_key_pairs_from above, and can end up owned/permissioned
# such that a later SFTP overwrite from this session gets a PermissionError (errno 13) --
# per-key filenames sidestep the clash entirely, and match how collect_key_pairs already
# names files on the remote side. ---
PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20  # paper used 150 for toeplitz/final-key; raise as time allows
CSV_SITE2 = RESULTS / "data" / "sdc_site2_toeplitz_finalkey_doseresponse.csv"

all_rows = []
for key_idx in range(5):
    meta_row = key_pairs_df_2.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    a_file = RESULTS / "data" / f"alice_sifted_bits_key{key_idx}_site2.json"
    b_file = RESULTS / "data" / f"bob_sifted_bits_key{key_idx}_site2.json"

    a_remote = f"results/alice_sifted_bits_key{key_idx}_site2.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}_site2.json"
    alice_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(b_file), f"qfabric/{b_remote}")
    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)
            # checkpoint per (key, prob, fault_type) block
            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2}")


NameError: name 'key_pairs_df_2' is not defined

In [2]:
import json

alice = slice_obj.get_node("alice")
recovered = []
for i in range(5):
    for remote_name in [f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json"]:
        local_path = RESULTS / "data" / remote_name
        stdout, _ = bob.execute(f"cat ~/qfabric/results/{remote_name}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            print(f"  {remote_name}: remote read failed, skipping")
            continue
        local_path.write_text(stdout)
        recovered.append((remote_name, len(data.get("alice_bits", data.get("bob_bits", [])))))
        print(f"  recovered {remote_name} ({recovered[-1][1]} bits)")

print(f"\nRecovered {len(recovered)} files.")

  recovered alice_sifted_bits_key0.json (3504 bits)
  recovered bob_sifted_bits_key0.json (3504 bits)
  recovered alice_sifted_bits_key1.json (3409 bits)
  recovered bob_sifted_bits_key1.json (3409 bits)
  recovered alice_sifted_bits_key2.json (3360 bits)
  recovered bob_sifted_bits_key2.json (3360 bits)
  recovered alice_sifted_bits_key3.json (3431 bits)
  recovered bob_sifted_bits_key3.json (3431 bits)
  recovered alice_sifted_bits_key4.json (3456 bits)
  recovered bob_sifted_bits_key4.json (3456 bits)

Recovered 10 files.


In [8]:
# --- Site-1 vs. site-2 CMH test, matched to your existing mock-vs-real methodology
# (same cmh_manual / stratify-by-prob pattern as notebooks/03_sdc_fault_models_and_statistics.ipynb) ---
from scipy.stats import chi2

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
        n = a + b + c + d
        if n < 2:
            continue
        row1, row2, col1, col2 = a + b, c + d, a + c, b + d
        E_a = row1 * col1 / n
        var_a = row1 * row2 * col1 * col2 / (n**2 * (n - 1))
        num += (a - E_a)
        den += var_a
    if den == 0:
        return np.nan, np.nan
    stat = num**2 / den
    return stat, 1 - chi2.cdf(stat, 1)


def build_tables_silent_both(site1_df, site2_df, fault_type):
    """One 2x2 table per firing-probability stratum, restricting BOTH sides
    to silent (verification-passed) mismatches, same as your mock-vs-real test."""
    tables = []
    for p in sorted(site2_df["prob"].unique()):
        s1 = site1_df[(site1_df["prob"] == p)]
        s2 = site2_df[(site2_df["prob"] == p) & (site2_df["fault_type"] == fault_type)]
        if len(s1) == 0 or len(s2) == 0:
            continue
        s1_mis = ((~s1["keys_match"].astype(bool)) & (s1["verification_passed"] == True)).sum()
        s2_mis = ((~s2["keys_match"].astype(bool)) & (s2["verification_passed"] == True)).sum()
        tables.append(np.array([[s1_mis, len(s1) - s1_mis], [s2_mis, len(s2) - s2_mis]]))
    return tables

site2 = pd.read_csv(str(CSV_SITE2))

# Your existing site-1 (NCSA-EDUKY) real-channel CSVs from the original submission:
site1_toeplitz = pd.read_csv(str(RESULTS / "data" / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv"))
site1_finalkey = pd.read_csv(str(RESULTS / "data" / "sdc_realchannel_finalkey_allkeys_asymptotic.csv"))

print("=== Site 1 (NCSA<->EDUKY) vs. Site 2 (NCSA<->UTAH), pooled CMH test ===\n")
for name, site1_df, ftype in [("toeplitz", site1_toeplitz, "toeplitz"),
                               ("final_key", site1_finalkey, "final_key")]:
    tables = build_tables_silent_both(site1_df, site2, ftype)
    stat, p = cmh_manual(tables)
    n1 = sum(t[0].sum() for t in tables)
    n2 = sum(t[1].sum() for t in tables)
    print(f"{name}: n_site1={n1}, n_site2={n2}, CMH stat={stat:.3f}, p={p:.4f} "
          f"-- {'SIGNIFICANT (site matters)' if p < 0.05 else 'not significant (consistent with Section 4.3)'}")


=== Site 1 (NCSA<->EDUKY) vs. Site 2 (NCSA<->UTAH), pooled CMH test ===

toeplitz: n_site1=1200, n_site2=580, CMH stat=35.533, p=0.0000 -- SIGNIFICANT (site matters)
final_key: n_site1=1200, n_site2=560, CMH stat=82.759, p=0.0000 -- SIGNIFICANT (site matters)


In [3]:
import pandas as pd
site1_meta = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata.csv")).iloc[:5]   # keys 0-4
site2_meta = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata_site2.csv"))
print("site 1 QBER:", sorted(site1_meta["qber"].tolist()))
print("site 2 QBER:", sorted(site2_meta["qber"].tolist()))

site 1 QBER: [0.0026178010471204, 0.0026737967914438, 0.0051948051948051, 0.0128205128205128, 0.0158311345646438]
site 2 QBER: [0.0102827763496143, 0.0105263157894736, 0.0153846153846153, 0.021108179419525, 0.0212765957446808]


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2

RESULTS = PROJECT_DIR / "results"

site1_toeplitz = pd.read_csv(str(RESULTS / "data" / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv"))
site1_finalkey = pd.read_csv(str(RESULTS / "data" / "sdc_realchannel_finalkey_allkeys_asymptotic.csv"))
site2 = pd.read_csv(str(RESULTS / "data" / "sdc_site2_toeplitz_finalkey_doseresponse.csv"))

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
        n = a + b + c + d
        if n < 2:
            continue
        row1, row2, col1, col2 = a + b, c + d, a + c, b + d
        E_a = row1 * col1 / n
        var_a = row1 * row2 * col1 * col2 / (n**2 * (n - 1))
        num += (a - E_a)
        den += var_a
    if den == 0:
        return np.nan, np.nan
    stat = num**2 / den
    return stat, 1 - chi2.cdf(stat, 1)


def add_ell_ratio(df):
    """The same quantity your model (Eq. 1) actually predicts P(D) from --
    ell/(ell+t_verify) -- computed per-trial from each row's OWN key's QBER,
    not from which site it came from."""
    df = df.copy()
    df["ell_ratio"] = df["secure_key_length"] / (df["secure_key_length"] + df["t_verify"])
    return df

site1_toeplitz = add_ell_ratio(site1_toeplitz)
site1_finalkey = add_ell_ratio(site1_finalkey)
site2 = add_ell_ratio(site2)

# Bin ell_ratio into quartiles computed over the POOLED range across both sites,
# so "same bin" means "the model predicts roughly the same P(D) regardless of
# which site the trial came from" -- this is what actually controls for the
# QBER confound, rather than requiring the raw QBER values themselves to match.
def build_tables_ell_controlled(site1_df, site2_df, fault_type, n_bins=4):
    s1 = site1_df  # site1_toeplitz/finalkey is already fault-type-specific
    s2 = site2_df[site2_df["fault_type"] == fault_type]

    pooled_ratio = pd.concat([s1["ell_ratio"], s2["ell_ratio"]])
    bin_edges = np.quantile(pooled_ratio, np.linspace(0, 1, n_bins + 1))
    bin_edges[0] -= 1e-9  # include the min value in the first bin

    tables, bin_report = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        for p in sorted(set(s1["prob"].unique()) & set(s2["prob"].unique())):
            s1_bin = s1[(s1["prob"] == p) & (s1["ell_ratio"] > lo) & (s1["ell_ratio"] <= hi)]
            s2_bin = s2[(s2["prob"] == p) & (s2["ell_ratio"] > lo) & (s2["ell_ratio"] <= hi)]
            if len(s1_bin) < 3 or len(s2_bin) < 3:
                continue  # too few trials in this (ell-bin, prob) cell to say anything
            s1_mis = ((~s1_bin["keys_match"].astype(bool)) & (s1_bin["verification_passed"] == True)).sum()
            s2_mis = ((~s2_bin["keys_match"].astype(bool)) & (s2_bin["verification_passed"] == True)).sum()
            tables.append(np.array([[s1_mis, len(s1_bin) - s1_mis], [s2_mis, len(s2_bin) - s2_mis]]))
            bin_report.append((round(lo, 3), round(hi, 3), p, len(s1_bin), len(s2_bin)))
    return tables, bin_report

print("=== Site 1 vs. Site 2, controlling for ell/(ell+t_verify) via stratification ===\n")
for name, site1_df, ftype in [("toeplitz", site1_toeplitz, "toeplitz"), ("final_key", site1_finalkey, "final_key")]:
    tables, bin_report = build_tables_ell_controlled(site1_df, site2, ftype)
    print(f"{name}: {len(tables)} usable (ell-bin, prob) strata with >=3 trials on both sides")
    for lo, hi, p, n1, n2 in bin_report:
        print(f"    ell_ratio in ({lo}, {hi}], prob={p}: n_site1={n1}, n_site2={n2}")
    if len(tables) == 0:
        print("    -> no usable strata at all; cannot say anything about a site effect from this data.\n")
        continue
    stat, p_val = cmh_manual(tables)
    n1_total = sum(t[0].sum() for t in tables)
    n2_total = sum(t[1].sum() for t in tables)
    print(f"    CMH stat={stat:.3f}, p={p_val:.4f} (n_site1={n1_total}, n_site2={n2_total}) "
          f"-- {'SIGNIFICANT even controlling for ell' if p_val < 0.05 else 'not significant once ell is controlled for'}\n")

In [3]:
site2 = pd.read_csv(str(RESULTS / "data" / "sdc_site2_toeplitz_finalkey_doseresponse.csv"))
print("columns:", site2.columns.tolist())
print()
print("row count:", len(site2))
print("error column value counts:")
print(site2["error"].value_counts(dropna=False) if "error" in site2.columns else "  (no 'error' column)")
print()
print("non_convergent value counts:")
print(site2["non_convergent"].value_counts(dropna=False) if "non_convergent" in site2.columns else "  (no 'non_convergent' column)")
print()
print("keys_match value counts:")
print(site2["keys_match"].value_counts(dropna=False) if "keys_match" in site2.columns else "  (no 'keys_match' column)")
print()
print("verification_passed value counts:")
print(site2["verification_passed"].value_counts(dropna=False) if "verification_passed" in site2.columns else "  (no 'verification_passed' column)")

columns: ['run', 'seed', 'error', 'raw_stdout', 'raw_stderr', 'keys_match', 'verification_passed', 'non_convergent', 'code_version', 'collection_timestamp', 'fault_type', 'prob', 'key_index', 'qber']

row count: 1140
error column value counts:
error
no bob output path found in stdout    1140
Name: count, dtype: int64

non_convergent value counts:
non_convergent
True    1140
Name: count, dtype: int64

keys_match value counts:
keys_match
NaN    1140
Name: count, dtype: int64

verification_passed value counts:
verification_passed
NaN    1140
Name: count, dtype: int64


In [4]:
print("=== raw_stdout ===")
print(site2["raw_stdout"].iloc[0])
print("\n=== raw_stderr ===")
print(site2["raw_stderr"].iloc[0])

=== raw_stdout ===
nan

=== raw_stderr ===
Traceback (most recent call last):
  File "/home/ubuntu/qfabric/scripts/bob_cascade_driver.py", line 25, in <module>
    from galois import GF2
ModuleNotFoundError: No module named 'galois'



In [5]:
# --- 1. Install the deps install_deps() actually missed, on both site-2 nodes ---
for node in [alice_2, bob_2]:
    _, stderr = node.execute(
        "cd ~/qfabric && source .venv/bin/activate && pip install --quiet -r requirements.txt",
        quiet=True,
    )
    check, _ = node.execute(
        "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
        quiet=True,
    )
    print(f"{node.get_name()}: {check.strip() or ('FAILED: ' + stderr[-500:])}")
    if "deps OK" not in check:
        raise RuntimeError(f"{node.get_name()} still missing deps after install -- stopping before wasting FABRIC time on the sweep.")

print("\ndeps fixed on both nodes, safe to proceed")

NameError: name 'alice_2' is not defined

In [16]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS = PROJECT_DIR / "results"

def show_key_data(metadata_csv_name):
    meta = pd.read_csv(str(RESULTS / "data" / metadata_csv_name))
    print(meta.to_string(index=False))

    # If the raw bit files are present, also show the true (measured) QBER
    # alongside the PE-sample estimate -- no modeling, just direct comparison.
    rows = []
    for _, row in meta.iterrows():
        idx = int(row["index"])
        n_bits = int(row["n_bits"])
        q_hat = float(row["qber"])
        alice_path = RESULTS / "data" / Path(row["alice_path"]).name if "alice_path" in row and pd.notna(row["alice_path"]) else None
        bob_path = RESULTS / "data" / Path(row["bob_path"]).name if "bob_path" in row and pd.notna(row["bob_path"]) else None
        true_qber = None
        if alice_path and bob_path and alice_path.exists() and bob_path.exists():
            with open(alice_path) as f:
                alice_bits = np.array(json.load(f)["alice_bits"], dtype=np.uint8)
            with open(bob_path) as f:
                bob_bits = np.array(json.load(f)["bob_bits"], dtype=np.uint8)
            true_qber = int(np.sum(alice_bits != bob_bits)) / n_bits
        rows.append({"index": idx, "n_bits": n_bits, "q_hat": q_hat, "true_qber": true_qber})

    print("\n--- q_hat vs. true qber (measured directly, no model) ---")
    print(pd.DataFrame(rows).to_string(index=False))

show_key_data("key_pairs_metadata.csv")       # or "key_pairs_metadata_site2.csv" for the Tokyo/NCSA keys

 index  m_sifted  k_pe  n_bits     qber                                                              alice_path                                                              bob_path  alice_raw_path  bob_raw_path
     0      3894   390    3504 0.012821 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key0.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key0.json             NaN           NaN
     1      3788   379    3409 0.015831 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key1.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key1.json             NaN           NaN
     2      3734   374    3360 0.002674 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key2.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key2.json             NaN           NaN
     3      3813   382    3431 0.002618 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key3.json /home/fabric/work/qkd-dependability/resu

In [17]:
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length

meta = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"))
rows = []
for _, row in meta.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2 = pd.DataFrame(rows)
table2.to_csv(str(RESULTS / "data" / "finitekey_vs_asymptotic_table2.csv"), index=False)
print(table2.to_string(index=False))

degenerate = table2[table2["ell_finite"] <= 1]
clean = table2[~table2.index.isin(degenerate.index)]
print(f"\nmean reduction (excluding {len(degenerate)} degenerate ell<=1 key(s)): {clean['reduction_pct'].mean():.2f}%")
if len(degenerate):
    print(f"degenerate keys (ell_finite<=1): {degenerate['key'].tolist()}")

 key  n_bits     qber  ell_asymptotic  ell_finite  reduction_pct
   0    3504 0.012821            2810          23      99.181495
   1    3409 0.015831            2608           1      99.961656
   2    3360 0.002674            3180         337      89.402516
   3    3431 0.002618            3251         363      88.834205
   4    3456 0.005195            3131         269      91.408496
   5    3455 0.010417            2877          87      96.976017
   6    3446 0.023499            2338           1      99.957228
   7    3388 0.010610            2813          66      97.653750
   8    3430 0.010471            2854          81      97.161878
   9    3351 0.013405            2663           1      99.962448

mean reduction (excluding 3 degenerate ell<=1 key(s)): 94.37%
degenerate keys (ell_finite<=1): [1, 6, 9]


In [25]:
import sys, json
from pathlib import Path
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()

slice_obj_2 = fablib.get_slice(name='qfabric-bb84-site2')  # adjust if you named it differently
alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")
bob_ip_2 = "10.10.1.2"  # fill in from setup_dataplane_ips output, or derive below if unknown
if bob_ip_2 is None:
    stdout, _ = bob_2.execute("ip -4 addr show | grep inet", quiet=True)
    print(stdout)  # pick the dataplane-side address (matches your 10.10.x.x convention)

key_pairs_df_2 = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata_site2.csv"))
print(key_pairs_df_2)

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
   index  m_sifted  k_pe  n_bits      qber  \
0      0      3855   386    3469  0.007772   
1      1      3791   380    3411  0.002632   
2      2      3894   390    3504  0.023077   
3      3      3711   372    3339  0.016129   
4      4      3882   389    3493  0.005141   

                                          alice_path  \
0  /home/fabric/work/qkd-dependability/results/al...   
1  /home/fabric/work/qkd-dependability/results/al...   
2  /home/fabric/work/qkd-dependability/results/al...   
3  /home/fabric/work/qkd-dependability/results/al...   
4  /home/fabric/work/qkd-dependability/results/al...   

                                            bob_path  \
0  /home/fabric/work/qkd-dependability/results/bo...   
1  /home/fabric/work/qkd-dependability/results/bo...   
2  /home/fabric/work/qkd-dependability/results/bo...   
3  /home/fabric/work/qkd-dependability/results/bo...   
4  /home/fabric/work/qkd-dependabi

In [20]:
for node in [alice_2, bob_2]:
    _, stderr = node.execute(
        "cd ~/qfabric && source .venv/bin/activate && pip install --quiet -r requirements.txt",
        quiet=True,
    )
    check, _ = node.execute(
        "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
        quiet=True,
    )
    print(f"{node.get_name()}: {check.strip() or ('FAILED: ' + stderr[-500:])}")
    if "deps OK" not in check:
        raise RuntimeError(f"{node.get_name()} still missing deps after install -- paste the stderr above.")

print("\ndeps fixed on both nodes, safe to proceed to Step 3")

alice: deps OK
bob: deps OK

deps fixed on both nodes, safe to proceed to Step 3


In [ ]:
from qne.cascade.sweep_utils import run_real_channel_trial

PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20  # raise once this runs cleanly end to end
CSV_SITE2 = RESULTS / "data" / "sdc_site2_toeplitz_finalkey_doseresponse.csv"

all_rows = []
first_trial_checked = False

for key_idx in range(len(key_pairs_df_2)):
    meta_row = key_pairs_df_2.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])
    a_remote = f"results/alice_sifted_bits_key{key_idx}_site2.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}_site2.json"

    # Confirm the per-key files are actually on the node before running trials
    check, _ = bob_2.execute(f"test -f ~/qfabric/{a_remote} && test -f ~/qfabric/{b_remote} && echo present || echo MISSING", quiet=True)
    if "present" not in check:
        a_local = RESULTS / "data" / f"alice_sifted_bits_key{key_idx}_site2.json"
        b_local = RESULTS / "data" / f"bob_sifted_bits_key{key_idx}_site2.json"
        alice_2.upload_file(str(a_local), f"qfabric/{a_remote}")
        bob_2.upload_file(str(a_local), f"qfabric/{a_remote}")
        bob_2.upload_file(str(b_local), f"qfabric/{b_remote}")
        print(f"  key{key_idx}: re-uploaded (was missing on node)")

    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

                if not first_trial_checked:
                    first_trial_checked = True
                    if result.get("error"):
                        raise RuntimeError(f"First trial failed: {result.get('error')!r}. "
                                            f"raw_stderr tail: {str(result.get('raw_stderr'))[-500:]}")
                    print(f"  first trial OK")

            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2}")


=== site2 key0: qber=0.0078 ===
  first trial OK
